Alright I got cellphone DB to work, now I need to do differential communicaton testing, 

My idea is unfortunately I have to loop through each individual and run cpdb_degs_analysis_method. Then, I collect all of the means.txt values, add sex, and then I'm going to have to loop through a lot of shit. I think its going to have to be a cluster by cluster, interaction by interation nested for loop where I coalesce the mean from each individual and run a linear regression. Before I do that, I will need to make sure it has a reasonable distribution and doesnt need some kind of special regression or non-parametric test. 

In [111]:
#libraries
import anndata  
import pandas as pd
import anndata as ad
import seaborn as sb
import scanpy as sc
import cellphonedb
import glob
import os
import sys

In [112]:
full_object = anndata.io.read_h5ad('C:/Users/Gabe/Desktop/RNA_object_human_names_anndata.h5ad')


In [113]:
#create an example subset object
#idk why this works in dot format but not 
subset_object =full_object[full_object.obs.individual=='T17D']

In [114]:
#t17d is the only one there
print(subset_object.obs.individual)

print(full_object.obs.shape)
print(subset_object.obs.shape)
### looks like it worked to me

T17D_AAACAGCCAAGGCCAA-1    T17D
T17D_AAACATGCAAGCTTAT-1    T17D
T17D_AAACATGCACAATGCC-1    T17D
T17D_AAACATGCAGCATGGA-1    T17D
T17D_AAACCAACAAAGGCCA-1    T17D
                           ... 
T17D_TTTGTGGCATTGTCCT-1    T17D
T17D_TTTGTGTTCAACAAGG-1    T17D
T17D_TTTGTGTTCATAAGCC-1    T17D
T17D_TTTGTTGGTGCCTCAC-1    T17D
T17D_TTTGTTGGTTTCGCCA-1    T17D
Name: individual, Length: 2795, dtype: object
(52517, 77)
(2795, 77)


Ok, the first thing I am going to do is write a loop that makes subset_objects

In [115]:
objects = {}
k =0
for i in set(full_object.obs.individual):
    if i == 'GH': # I dont want to waste time analyzing this
        continue
    print(i)
    key =  f"{'obj_'}{i}" #dynamically name object
    value = full_object[full_object.obs.individual==i] #subset object
    objects[key] = value 
    k += 1



T13D
A12D
T5D
A22D
B23F
C15M
T17D
C21D
B21M
B23M
C14F
T19D
T14D
C13F
C13M
T11D
B21F
D4F
D4M
C15F
A25D
C14M


In [116]:
#I think it worked
objects['obj_A12D'].obs.individual

A12D_AAACAGCCAAACTAAG-1    A12D
A12D_AAACAGCCAAAGCTAA-1    A12D
A12D_AAACAGCCATTAGGTT-1    A12D
A12D_AAACATGCAAACGGGC-1    A12D
A12D_AAACCGAAGAACCTGT-1    A12D
                           ... 
A12D_TTTGTGTTCACGCGGT-1    A12D
A12D_TTTGTGTTCTACCTGC-1    A12D
A12D_TTTGTTGGTCGTTACT-1    A12D
A12D_TTTGTTGGTCTAACAG-1    A12D
A12D_TTTGTTGGTTGAGGTC-1    A12D
Name: individual, Length: 3993, dtype: object

Ok, next, I want to loop through the cellphone function, I will need to make sure to also make a dynamic path just to stop any mess in my folder

Wait a second, will it work with the URL meta data paths etc? I think because everything I care about is in the anndata object, I can write the relevant files at each subset iteration

In [ ]:
### SUCCEEDED, DONT NEED TO RUN AGAIN
## Write files
import hdf5plugin
from scipy.sparse import csr_matrix

for i in set(full_object.obs.individual):
    if i == 'GH': # I dont want to waste time analyzing this
        continue
    key =  f"{'obj_'}{i}" #dynamically name object
    relevant_object = objects[key]
    temp_meta_data = relevant_object.obs[['cell_barcode', 'harmony.wnn_res0.4_clusters']]
    
    cells = {
    "Cell" : temp_meta_data['cell_barcode'],
    "harmony.wnn_res0.4_clusters" : temp_meta_data['harmony.wnn_res0.4_clusters']
    
    }
    cells = pd.DataFrame(cells)
    cells2 = cells[['Cell','harmony.wnn_res0.4_clusters']]

    filename = f"{"A:/CellPhoneDB 030225/"}{i}{"/cluster_labels_metadata.tsv"}"
    # I need to add a line here to make the folder
    
    #file_path = os.makedirs(f"{"A:/CellPhoneDB 030225/"}{i}{"/"}")

    cells2.to_csv(filename, sep="\t") 

    folder_path = f"{"A:/CellPhoneDB 030225/"}{i}{"/normalised_log_counts.h5ad"}"

    relevant_object.X = csr_matrix(relevant_object.X)
    relevant_object.write_h5ad(folder_path)


#### Succeeded, dont need to run again
#counts_file_path: (mandatory) paths to normalized counts file (not z-transformed), either in text format or h5ad (recommended) normalised_log_counts.h5ad.
obj.X = csr_matrix(obj.X)
obj.X

#import hdf5plugin
#obj.write_h5ad('A:/CellPhoneDB 030225/normalised_log_counts')

c:\Users\Gabe\AppData\Local\Programs\Python\Python312\Lib\site-packages\anndata\_core\anndata.py:617: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  warnings.warn(msg, FutureWarning, stacklevel=1)
c:\Users\Gabe\AppData\Local\Programs\Python\Python312\Lib\site-packages\anndata\_core\anndata.py:617: FutureWarning: You are attempting to set `X` to a matrix on a view which has non-unique indices. The resulting `adata.X` will likely not equal the value to which you set it. To avoid this potential issue, please make a copy of the data first. In the future, this operation will throw an error.
  warnings.warn(msg, FutureWarning, stacklevel=1)
c:\Users\Gabe\AppData\Local\Programs\Python\Python312\Lib\site-packages\anndata\_core\anndata.py:617: Future

In [119]:
for i in set(full_object.obs.individual):
    if i == 'GH': # I dont want to waste time analyzing this
        continue
    key =  f"{'obj_'}{i}" #dynamically name object
    relevant_object = objects[key]
    temp_meta_data = relevant_object.obs[['cell_barcode', 'harmony.wnn_res0.4_clusters']]
    
    cells = {
    "Cell" : temp_meta_data['cell_barcode'],
    "cell_type" : temp_meta_data['harmony.wnn_res0.4_clusters']
    
    }
    cells = pd.DataFrame(cells)
    cells2 = cells[['Cell','cell_type']]

    filename = f"{"A:/CellPhoneDB 030225/"}{i}{"/cluster_labels_metadata2.tsv"}"
    # I need to add a line here to make the folder
    
    cells2.to_csv(filename, sep="\t") 


In [135]:
 from cellphonedb.src.core.methods import cpdb_degs_analysis_method

for i in set(full_object.obs.individual):
    cpdb_file_path = 'A:/CellPhoneDB 030225/v5.0.0/cellphonedb.zip' # Dont change
    degs_file_path =   f"{"A:/CellPhoneDB 030225/"}{i}{"/DEGs.tsv"}" 

    meta_file_path =  f"{"A:/CellPhoneDB 030225/"}{i}{"/human_named_meta.tsv"}"
    counts_file_path  =  f"{"A:/CellPhoneDB 030225/"}{i}{"/normalised_log_counts.h5ad"}"
    out_path =  f"{"A:/CellPhoneDB 030225/"}{i}"

    cpdb_results = cpdb_degs_analysis_method.call(
        cpdb_file_path = cpdb_file_path,                            # mandatory: CellphoneDB database zip file.
        meta_file_path = meta_file_path,                            # mandatory: tsv file defining barcodes to cell label.
        counts_file_path = counts_file_path,                        # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
        degs_file_path = degs_file_path,                            # mandatory: tsv file with DEG to account.
        counts_data = 'gene_name',                                # defines the gene annotation in counts matrix.
        score_interactions = True,                                  # optional: whether to score interactions or not. 
        threshold = 0.1,                                            # defines the min % of cells expressing a gene for this to be employed in the analysis.
        result_precision = 3,                                       # Sets the rounding for the mean values in significan_means.
        separator = '_',                                            # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
        debug = True,                                              # Saves all intermediate tables emplyed during the analysis in pkl format.
        output_path = out_path,                                     # Path to save results
        output_suffix = None,                                       # Replaces the timestamp in the output files by a user defined string in the  (default: None)
        threads = 25
        ) 

[ ][CORE][04/03/25-15:36:51][INFO] [Cluster DEGs Analysis] Threshold:0.1 Precision:3
Reading user files...
The following user files were loaded successfully:
A:/CellPhoneDB 030225/T13D/normalised_log_counts.h5ad
A:/CellPhoneDB 030225/T13D/human_named_meta.tsv
A:/CellPhoneDB 030225/T13D/DEGs.tsv
[ ][CORE][04/03/25-15:36:52][INFO] Running Real Analysis
[ ][CORE][04/03/25-15:36:52][INFO] Running DEGs-based Analysis
[ ][CORE][04/03/25-15:36:52][INFO] Saving intermediate data to file debug_intermediate.pkl
[ ][CORE][04/03/25-15:36:52][INFO] Building results
[ ][CORE][04/03/25-15:36:52][INFO] Scoring interactions: Filtering genes per cell type..


100%|██████████| 30/30 [00:00<00:00, 361.31it/s]

[ ][CORE][04/03/25-15:36:52][INFO] Scoring interactions: Calculating mean expression of each gene per group/cell type..



100%|██████████| 30/30 [00:00<00:00, 881.91it/s]
c:\Users\Gabe\AppData\Local\Programs\Python\Python312\Lib\site-packages\cellphonedb\utils\scoring_utils.py:138: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  matrix[index_name].replace(to_replace=id2name, inplace=True)


[ ][CORE][04/03/25-15:36:53][INFO] Scoring interactions: Calculating scores for all interactions and cell types..


  0%|          | 0/910 [00:06<?, ?it/s]


KeyError: 'id'

In [ ]:
import pickle as pkl

with open('A:/CellPhoneDB 030225/T13D/debug_intermediate.pkl', 'rb') as f:
    debug = pkl.load(f)

debug

{'meta':                         cell_type
 T13D_AAACAGCCATGAGCAG-1        23
 T13D_AAACCAACAGGAACCA-1         4
 T13D_AAACCAACATGTCGCG-1         3
 T13D_AAACCGAAGAGAAGGG-1         7
 T13D_AAACCGAAGCTCCTTA-1         7
 ...                           ...
 T13D_TTTGTGGCACGAACAG-1         1
 T13D_TTTGTGTTCATTGTCT-1         3
 T13D_TTTGTTGGTGCGCATG-1         3
 T13D_TTTGTTGGTTAGTGAT-1         7
 T13D_TTTGTTGGTTGGTGAC-1         2
 
 [1982 rows x 1 columns],
 'genes':       id_gene          ensembl gene_name hgnc_symbol  protein_id  id_protein  \
 0           0  ENSG00000196502   SULT1A1     SULT1A1        1154        1154   
 1           1  ENSG00000154127   UBASH3B     UBASH3B        1113        1113   
 2           2  ENSG00000128039    SRD5A3      SRD5A3        1146        1146   
 3           3  ENSG00000105398   SULT2A1     SULT2A1        1151        1151   
 4           4  ENSG00000277893    SRD5A2      SRD5A2        1130        1130   
 ...       ...              ...       ...        

In [133]:
metadata_partial = pd.read_csv(meta_file_path, sep = '\t')
metadata_partial.head()

FileNotFoundError: [Errno 2] No such file or directory: 'A:/CellPhoneDB 030225/T13Dhuman_named_meta.tsv'

In [134]:
f"{"A:/CellPhoneDB 030225/"}{i}{"human_named_meta.tsv"}"

'A:/CellPhoneDB 030225/T13Dhuman_named_meta.tsv'

In [ ]:

metadata_full = pd.read_csv('A:/CellPhoneDB 030225/human_named_meta.tsv', sep = '\t')
metadata_full

,Cell,harmony.wnn_res0.4_clusters
0,T17D_AAACAGCCAAGGCCAA-1,7
1,D4M_AAACAGCCACCAGCAT-1,2
2,T19D_AAACAGCCACGTAATT-1,0
3,D4M_AAACATGCAACTAACT-1,1
4,T17D_AAACATGCAAGCTTAT-1,5
...,...,...
52512,T11D_TTTGTTGGTAAATTGC-1,15
52513,T11D_TTTGTTGGTAATTAGC-1,4
52514,T11D_TTTGTTGGTCCTTAGT-1,0
52515,T11D_TTTGTTGGTGACATAT-1,2


In [ ]:
adata_full = anndata.read_h5ad('A:/CellPhoneDB 030225/normalised_log_counts')
adata_full.obs_keys

<bound method AnnData.obs_keys of AnnData object with n_obs × n_vars = 52517 × 11115
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'TSS.enrichment', 'TSS.percentile', 'peak_region_fragments', 'log10nfragments', 'pct_reads_in_peaks', 'nucleosome_signal', 'nucleosome_percentile', 'individual', 'pct.mt', 'pct.rb', 'pct.mt.rb', 'pct.cd', 'pct.nuclear', 'pct.ambient', 'log10GenesPerUMI', 'log10PeaksPerUMI', 'genes_per_umi', 'peaks_per_umi', 'pct_s.genes', 'pct_g2m.genes', 'percent_top50_genes', 'percent_top100_genes', 'nuc_prep_batch', 'Tank', 'Trigger', 'Prev_tank', 'Condition', 'Status', 'Status_Long', 'Time_Day_2', 'Behaviors_Day_2', 'Change_Length', 'Change_Mass', 'Log_11KT', 'Average_Area_2.5x', 'Total_Slides_With_Gonad', 'Estimated_Volume_2.5x', 'Log10_Volume', 'Percent_Testicular', 'Testicular_Estimate', 'Log10_Testicular_Estimate', 'pct.bad_nuc', 'pct.good_nuc', 'S.Score', 'G2M.Score', 'Phase', 'median_UMI_count', 'median_gene_count', 'median_p

In [ ]:
adata_partial = anndata.read_h5ad(counts_file_path)
adata_partial.obs_keys

<bound method AnnData.obs_keys of AnnData object with n_obs × n_vars = 1982 × 11115
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'TSS.enrichment', 'TSS.percentile', 'peak_region_fragments', 'log10nfragments', 'pct_reads_in_peaks', 'nucleosome_signal', 'nucleosome_percentile', 'individual', 'pct.mt', 'pct.rb', 'pct.mt.rb', 'pct.cd', 'pct.nuclear', 'pct.ambient', 'log10GenesPerUMI', 'log10PeaksPerUMI', 'genes_per_umi', 'peaks_per_umi', 'pct_s.genes', 'pct_g2m.genes', 'percent_top50_genes', 'percent_top100_genes', 'nuc_prep_batch', 'Tank', 'Trigger', 'Prev_tank', 'Condition', 'Status', 'Status_Long', 'Time_Day_2', 'Behaviors_Day_2', 'Change_Length', 'Change_Mass', 'Log_11KT', 'Average_Area_2.5x', 'Total_Slides_With_Gonad', 'Estimated_Volume_2.5x', 'Log10_Volume', 'Percent_Testicular', 'Testicular_Estimate', 'Log10_Testicular_Estimate', 'pct.bad_nuc', 'pct.good_nuc', 'S.Score', 'G2M.Score', 'Phase', 'median_UMI_count', 'median_gene_count', 'median_pe

In [ ]:
deg_full = pd.read_csv('A:/CellPhoneDB 030225/human_named_DEGs.tsv', sep = '\t')
deg_full.head()

,cluster,gene,p_val_adj,p_val,avg_log2FC,pct.1,pct.2
0,7,KCNIP4,0.0,0.0,2.287049,0.947,0.311
1,7,NXPH1,0.0,0.0,2.644073,0.762,0.166
2,7,MCAM,0.0,0.0,2.361951,0.773,0.196
3,7,ST18,0.0,0.0,2.705234,0.662,0.105
4,7,EBF1,0.0,0.0,3.570834,0.659,0.112


In [ ]:
deg_partial= pd.read_csv(degs_file_path, sep = '\t')
deg_partial.head()

,cluster,gene,p_val_adj,p_val,avg_log2FC,pct.1,pct.2
0,7,NXPH1,3.521549e-77,3.168285e-81,2.684349,0.801,0.205
1,7,BCAR3,6.737401e-74,6.061539e-78,4.554481,0.335,0.020
2,7,ST18,5.197751e-65,4.676339e-69,2.841636,0.608,0.119
3,7,IGF1,9.135556e-64,8.219123e-68,4.622411,0.392,0.044
4,7,KCNIP4,3.886192e-62,3.496349e-66,1.914011,0.949,0.404


In [130]:
adata = anndata.read_h5ad(counts_file_path)
adata.shape

list(adata.obs.index).sort() == list(metadata_partial['Cell']).sort()


True

In [131]:
print(adata.X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2302373 stored elements and shape (1982, 11115)>
  Coords	Values
  (0, 0)	4.521699559359188
  (0, 1)	5.056499136798742
  (0, 2)	3.6806335388906586
  (0, 3)	2.057245242007275
  (0, 5)	2.8287071404115056
  (0, 6)	4.444619024244306
  (0, 7)	3.066803662830968
  (0, 8)	2.057245242007275
  (0, 9)	3.789202279141585
  (0, 10)	2.8287071404115056
  (0, 11)	2.6843564828386732
  (0, 12)	2.8287071404115056
  (0, 13)	2.5155955852053373
  (0, 14)	2.6843564828386732
  (0, 15)	3.066803662830968
  (0, 16)	2.312454338748358
  (0, 17)	4.141080324247916
  (0, 18)	2.057245242007275
  (0, 19)	2.312454338748358
  (0, 21)	3.1674963239969114
  (0, 22)	2.312454338748358
  (0, 23)	3.9327173116363507
  (0, 24)	1.713724147892835
  (0, 25)	2.057245242007275
  (0, 26)	2.312454338748358
  :	:
  (1981, 6770)	2.6061689179569254
  (1981, 6867)	2.6061689179569254
  (1981, 6896)	2.6061689179569254
  (1981, 7039)	2.6061689179569254
  (1981, 7131)	2.60616891795692